# Heart Disease Prediction: RAG Explainer with an LLM

**Goal:** turn the logistic regression's prediction for one patient into a plain-language explanation that is *grounded* in project documentation, not invented by the LLM.

**Pipeline (retrieval-augmented generation)**
1. **Predict:** the final model from `heart_disease_prediction.ipynb` (0.85 accuracy, 0.91 ROC-AUC) scores the patient.
2. **Attribute:** each feature's contribution = coefficient x (patient value - average patient); the top 3 are the main reasons.
3. **Retrieve:** the knowledge base (`knowledge_base/*.md`: feature guide built from the data, model guide, metrics guide) is split into 23 passages and embedded with `all-MiniLM-L6-v2`; the passages closest to the patient's top factors are retrieved.
4. **Explain (final grounded design):** a template states the facts (risk score, each factor's value and direction) so they are always correct, and each factor's meaning is quoted from its retrieved definition.
5. **Chat:** follow-up questions are answered by an LLM (local `Qwen2.5-1.5B-Instruct` via Hugging Face transformers; Ollama or OpenAI optional) using the facts and retrieved passages, labelled as AI-generated.
6. **Evaluate:** retrieval hit rate, and a check of the LLM's chat answers for numbers that do not appear in its context (hallucinated statistics). Four earlier designs are compared in the evaluation history. Earlier free-text versions are compared in the evaluation history below.

> Screening aid for a portfolio project, not medical advice.

In [1]:
# One-time setup (see GENAI_SETUP.md):
#   pip install sentence-transformers transformers torch gradio requests
# Default LLM: Qwen2.5-1.5B-Instruct via Hugging Face transformers (free, runs on CPU, ~3 GB download).
import os, json
os.environ.setdefault('LLM_BACKEND', 'hf')        # or 'ollama', 'openai' (needs OPENAI_API_KEY), or 'none'
import pandas as pd
import rag_core as R
from sklearn.metrics import accuracy_score, roc_auc_score

model, X_train, X_test, y_test = R.train_model()
print('Test accuracy:', round(accuracy_score(y_test, model.predict(X_test)), 3),
      '| ROC-AUC:', round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3))

Test accuracy: 0.846 | ROC-AUC: 0.912


In [2]:
retriever = R.Retriever()
print(len(retriever.chunks), 'knowledge-base passages, embedded with', retriever.embedder.backend)
for p in retriever.search('What does ST depression mean?', k=3):
    print(f"  {p['score']:.3f}  {p['title']}  ({p['source']})")

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2451.89it/s]

23 knowledge-base passages, embedded with sentence-transformers
  0.629  ST depression (oldpeak)  (feature_guide.md)
  0.413  ST slope (slope)  (feature_guide.md)
  0.275  Resting ECG result (restecg)  (feature_guide.md)


## Explain three patients: high, low and borderline risk

In [3]:
probs = model.predict_proba(X_test)[:, 1]
picks = {'High risk': probs.argmax(), 'Low risk': probs.argmin(), 'Borderline': abs(probs - 0.5).argmin()}
examples = []
for label, i in picks.items():
    out = R.explain(model, X_train, retriever, X_test.iloc[i])
    examples.append({'case': label, 'risk': round(out['risk'], 3), 'actual_label': int(y_test.iloc[i]),
                     'factors': out['factors'], 'retrieved': [p['title'] for p in out['passages']], 'answer': out['answer'], 'llm_text': out['llm_text']})
    print(f"=== {label}: risk {out['risk']:.2f} (actual label {y_test.iloc[i]}) ===")
    print('Top factors:', ', '.join(f"{f['name']}={f['value']} ({f['direction']})" for f in out['factors']))
    print('Retrieved:', [p['title'] for p in out['passages']])
    print(out['answer'], '\n')

=== High risk: risk 0.98 (actual label 1) ===
Top factors: Sex=0 (raises risk), Maximum heart rate=179 (raises risk), ST depression=0 (raises risk)
Retrieved: ['ST depression (oldpeak)', 'Maximum heart rate (thalach)', 'ST slope (slope)', 'How to read a risk score']
**Risk score: 0.98**, which the model rates as high. The main reasons:
- **Sex: female (coded 0)**, which pushed the risk score up (contribution +0.66).
- **Maximum heart rate: 179 bpm**, which pushed the risk score up (contribution +0.44).
- **ST depression: 0 mm**, which pushed the risk score up (contribution +0.44).

**What these factors mean** _(from the project knowledge base)_:
- **Sex:** Biological sex, coded 1 = male, 0 = female.
- **Maximum heart rate:** Maximum heart rate (beats per minute) reached during an exercise stress test. Maximum achievable heart rate naturally declines with age.
- **ST depression:** ST-segment depression on the ECG induced by exercise relative to rest, in millimetres. ST depression on an 

=== Borderline: risk 0.50 (actual label 0) ===
Top factors: Maximum heart rate=195 (raises risk), Resting blood pressure=192 (lowers risk), Thalassemia test result=3 (lowers risk)
Retrieved: ['Thalassemia test result (thal)', 'Maximum heart rate (thalach)', 'Resting blood pressure (trestbps)', 'What the model is']
**Risk score: 0.50**, which the model rates as uncertain (close to 50/50). The main reasons:
- **Maximum heart rate: 195 bpm**, which pushed the risk score up (contribution +0.68).
- **Resting blood pressure: 192 mm Hg**, which pushed the risk score down (contribution -0.57).
- **Thalassemia test result: code 3**, which pushed the risk score down (contribution -0.54).

**What these factors mean** _(from the project knowledge base)_:
- **Maximum heart rate:** Maximum heart rate (beats per minute) reached during an exercise stress test. Maximum achievable heart rate naturally declines with age.
- **Resting blood pressure:** Resting systolic blood pressure in mm Hg, measured on 

In [4]:
# Follow-up questions answered by the LLM (RAG chat), checked for hallucinated numbers
questions = ['Why is this patient borderline, and how confident is the model?',
             'What does ST depression mean?',
             'Which factor pushed the risk score up the most?',
             'What is the thalassemia test result?',
             'How accurate is this model overall?']
chat = []
for q in questions:
    out = R.explain(model, X_train, retriever, X_test.iloc[picks['Borderline']], question=q)
    context = R.facts_block(out['risk'], out['factors']) + ' ' + ' '.join(p['text'] for p in out['passages'])
    extra = R.unsupported_numbers(out['llm_text'], context)
    chat.append({'question': q, 'answer': out['llm_text'], 'unsupported_numbers': extra})
    print(f'Q: {q}\nA: {out["llm_text"]}\n   unsupported numbers: {extra or "none"}\n')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:  47%|████▋     | 160/338 [00:00<00:00, 1590.08it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1707.51it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


...

[transformers] Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Which factor pushed the risk score up the most?
A: The maximum heart rate contributed the most to pushing the risk score up, with a contribution of +0.68.
   unsupported numbers: none



[transformers] Both `max_new_tokens` (=260) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the thalassemia test result?
A: The thalassemia test result was coded as 3.
   unsupported numbers: none



Q: How accurate is this model overall?
A: The model has an accuracy of 0.85 on the 91-patient test set and achieves 0.91 ROC-AUC, indicating good performance.
   unsupported numbers: none



## Evaluation
**Retrieval hit rate:** for all 91 test patients, the share of top-3 factors whose knowledge-base passage is among the 4 retrieved. **Direction accuracy:** whether each factor's direction is stated correctly in the explanation. **Hallucinated numbers:** numbers in the LLM's chat answers that do not appear in the facts or passages it was given.

In [5]:
hits = total = 0
for i in range(len(X_test)):
    risk, factors = R.explain_patient(model, X_train, X_test.iloc[i])
    query = ' '.join(f"{f['name']} ({f['feature']})" for f in factors) + ' risk score'
    titles = [p['title'] for p in retriever.search(query, k=4)]
    for f in factors:
        total += 1; hits += any(f"({f['feature']})" in t for t in titles)
hit_rate = hits / total
print(f'Retrieval hit rate @4: {hit_rate:.1%} ({hits}/{total} factor passages found)')

faith = [R.check_faithfulness(R.facts_block(ex['risk'], ex['factors']), ex['factors'])[0] for ex in examples]   # facts section only
clean_chat = sum(len(c['unsupported_numbers']) == 0 for c in chat)
print(f"Explanation directions correct: {sum(faith) / len(faith):.0%} (template facts)")
print(f"Chat answers with no hallucinated numbers: {clean_chat}/{len(chat)}")

Retrieval hit rate @4: 92.7% (253/273 factor passages found)
Explanation directions correct: 100% (template facts)
Chat answers with no hallucinated numbers: 5/5


In [6]:
os.makedirs('results', exist_ok=True)
json.dump({'embedding_model': retriever.embedder.backend, 'llm_backend': os.environ['LLM_BACKEND'],
           'llm_model': {'hf': os.environ.get('HF_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct'), 'ollama': os.environ.get('OLLAMA_MODEL', 'llama3.2:3b'),
                         'openai': os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}.get(os.environ['LLM_BACKEND'], 'none'),
           'design': 'grounded v5 (template facts + quoted definitions; LLM for chat)', 'kb_passages': len(retriever.chunks),
           'retrieval_hit_rate_at_4': round(hit_rate, 3), 'explanation_direction_accuracy': round(sum(faith) / len(faith), 3),
           'chat_answers_without_hallucinated_numbers': f'{clean_chat}/{len(chat)}', 'examples': examples, 'chat': chat},
          open('results/rag_results.json', 'w'), indent=1, default=str)
print('Saved results/rag_results.json')

Saved results/rag_results.json


## Evaluation history: why the hybrid design
Earlier versions let the LLM write the whole explanation. A manual review of the same three patients plus one follow-up question found:

| Version | Design | Errors found by manual review |
|---|---|---|
| v1 | Free text, raw codes in the prompt | 4 factual errors (called a female patient "male", misread ST depression, invented a "mild abnormality" reading of thal, contradicted the 50% score in the follow-up); safety note missing in 1 of 3 |
| v2 | Free text, readable values ("female"), strict rules | Sex and directions fixed, safety note 3 of 3; still about 5 errors (misread numeric values, e.g. "no significant ST depression" for 4 mm; invented statistics, e.g. "62% accurate") |
| v3 | Hybrid: template facts + LLM context from all retrieved passages | Facts 100% correct; but the LLM's context drifted (e.g. described maximum heart rate as measured "at rest", invented claims about ST slope, explained factors that were not in the top three) |
| v4 | Hybrid: template facts + LLM paraphrases only each top factor's retrieved definition | No invented numbers in explanations, but it skipped factors (covered 1 of 3 for one patient) and invented details ("blood pressure when you first wake up"); chat answers invented statistics |
| v5 | **Grounded: template facts + quoted definitions; LLM only for follow-up chat, labelled and checked** | Results in the evaluation above |

Lesson: a small local LLM (1.5B parameters) is fluent but unreliable with numbers and specifics, so the final design keeps it away from anything that must be correct: facts and definitions come from code and the knowledge base, and the LLM answers follow-up questions with a visible label and an automatic check for hallucinated numbers. A larger model (e.g. Llama 3.2 3B via Ollama or GPT-4o-mini) is the next step for better chat answers.

## Try the chat app
Run `python app.py` in this folder and open the local link it prints. Pick a test patient, click **Explain**, then ask follow-up questions.

## Conclusion
*(Fill in after running; Claude can update this and the report from `results/rag_results.json`.)*